# Réplica de vcMSA

Este cuaderno reproduce una ejecución del método descrito por McWhite, Armour-Garb y Singh (2023), usando el código oficial de `vcMSA`.

La configuración principal sigue el artículo y el repositorio de benchmarking de los autores:

- modelo `Rostlab/prot_t5_xl_uniref50`;
- concatenación de las últimas 16 capas: 16 x 1024 = 16384 componentes por aminoácido;
- 10 caracteres `X` de padding en cada extremo;
- corrección por lote con ComBat;
- similitud coseno y búsqueda con FAISS;
- reciprocal best hits, maximum non-crossing matching y maximum increasing subsequence;
- familia `csp` de QuanTest2 como réplica manejable del benchmark;
- cálculo de Total Column Score (TCS) y comparación entre vcMSA y el Código Propio.

> Alcance: esto reproduce una familia real del benchmark y toda la tubería de vcMSA. No ejecuta automáticamente las 147 familias del artículo. Tampoco pretende igualdad bit a bit con el servidor A100 de 2023, porque Colab actual utiliza versiones modernas de Python, CUDA y PyTorch. Todas esas diferencias quedan registradas.

Fuentes: [artículo](https://pmc.ncbi.nlm.nih.gov/articles/PMC10538487/), [vcMSA](https://github.com/clairemcwhite/vcmsa) y [benchmark QuanTest2](https://github.com/clairemcwhite/nf-benchmark-vcmsa).

## Antes de ejecutar

1. En Colab, seleccione **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**.
2. Ejecute las celdas en orden.
3. La primera ejecución descargará ProtT5-XL-UniRef50 y puede tardar varios minutos.
4. Al final se descargará un ZIP con comandos, versiones, parches, tiempos, RAM/VRAM, alineamientos, métricas y hashes.

Si Colab desconecta la sesión durante la descarga o el embedding, vuelva a ejecutar desde el inicio.

In [17]:
# Dependencias modernas compatibles con Colab. No se reemplaza el PyTorch/CUDA del runtime.
!apt-get -qq update
!pip -q install "transformers>=4.41,<5" "sentencepiece>=0.2,<1" "biopython>=1.83,<2" "faiss-cpu>=1.9,<2" "python-igraph>=0.11,<1" "combat==0.3.3" "psutil>=5.9,<8" "pynvml>=11,<13" "numba>=0.58,<1"
print('Dependencias instaladas.')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Dependencias instaladas.


## 1. Configuración de la réplica

La opción recomendada es `DATASET_MODE = "quantest2"`. Para una prueba preliminar más pequeña puede usarse `"zf10"`, pero en ese caso no habrá referencia estructural ni TCS.

In [18]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, os, platform, shlex, shutil, subprocess, sys, threading, time, zipfile

import numpy as np
import pandas as pd
import psutil
import torch
from Bio import AlignIO, SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ---------- Parámetros principales ----------
DATASET_MODE = 'quantest2'       # 'quantest2' o 'zf10'
QUANTEST2_FAMILY = 'csp'         # Familia real del benchmark usada por defecto
INCLUDE_EMBEDDINGS_IN_ZIP = False  # El pickle puede ser grande

VCMSA_COMMIT = 'bef34fe085dad28ea5d8743baf02773e0a15f0af'
BENCHMARK_COMMIT = 'e35f03d5e4afdbbd4638cd9ee17d5690cfda971a'
MODEL_NAME = 'Rostlab/prot_t5_xl_uniref50'
LAYERS = list(range(-16, 0))
PADDING = 10
SEQSIM_THRESHOLD = 0.9
USE_BATCH_CORRECTION = True
USE_MNCM = True
USE_MIS = True
USE_OUTLIER_EXCLUSION = True

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
BASE_DIR = Path('/content/vcmsa_paper_replica')
VCMSA_DIR = BASE_DIR / 'vcmsa'
BENCHMARK_DIR = BASE_DIR / 'nf-benchmark-vcmsa'
EVIDENCE_DIR = BASE_DIR / 'runs' / RUN_ID
CACHE_DIR = BASE_DIR / 'cache' / RUN_ID
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('No se detectó GPU. Active una GPU T4 en el tipo de entorno de ejecución y reinicie.')

print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('Directorio de evidencias:', EVIDENCE_DIR)

GPU: Tesla T4
PyTorch: 2.11.0+cu128
Directorio de evidencias: /content/vcmsa_paper_replica/runs/20260722T143442Z


## 2. Descarga versionada del código y los datos

Se fija un commit de vcMSA cuyo código fuente coincide con el estado final de mayo de 2023; los dos commits posteriores solo modificaron el README. También se fija el commit del repositorio de benchmarking que incorporó QuanTest2.

In [19]:
def command_output(cmd, cwd=None):
    return subprocess.run(cmd, cwd=cwd, check=True, text=True, capture_output=True).stdout.strip()

def clone_at_commit(url, destination, commit):
    if not destination.exists():
        subprocess.run(['git', 'clone', '--quiet', url, str(destination)], check=True)
        subprocess.run(['git', '-C', str(destination), 'checkout', '--quiet', commit], check=True)
    head = command_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'])
    if head != commit:
        raise RuntimeError(f'{destination} ya existe en el commit {head}, pero se esperaba {commit}. Reinicie el runtime para una ejecución limpia.')
    return head

vcmsa_head = clone_at_commit('https://github.com/clairemcwhite/vcmsa.git', VCMSA_DIR, VCMSA_COMMIT)
benchmark_head = clone_at_commit('https://github.com/clairemcwhite/nf-benchmark-vcmsa.git', BENCHMARK_DIR, BENCHMARK_COMMIT)
print('vcMSA:', vcmsa_head)
print('Benchmark:', benchmark_head)

vcMSA: bef34fe085dad28ea5d8743baf02773e0a15f0af
Benchmark: e35f03d5e4afdbbd4638cd9ee17d5690cfda971a


## 3. Parches mínimos para Colab actual

Estos parches no sustituyen el algoritmo de alineamiento. Solo eliminan dependencias de perfilado, desactivan un trazado global que distorsiona los tiempos, permiten FAISS en CPU y hacen que `--cpu_only` se respete. El diff exacto se guardará como evidencia.

In [20]:
def replace_once(path, old, new, label):
    text = path.read_text(encoding='utf-8')
    if new in text:
        print(f'Ya aplicado: {label}')
        return
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f'No se pudo aplicar {label}: se esperó 1 coincidencia y se encontraron {count}.')
    path.write_text(text.replace(old, new), encoding='utf-8')
    print(f'Aplicado: {label}')

bin_file = VCMSA_DIR / 'bin' / 'vcmsa'
embed_file = VCMSA_DIR / 'src' / 'vcmsa' / 'vcmsa_embed.py'
utils_file = VCMSA_DIR / 'src' / 'vcmsa' / 'vcmsa_utils.py'

replace_once(
    bin_file,
    'from scalene import scalene_profiler',
    'try:\n    from scalene import scalene_profiler\nexcept ImportError:\n    scalene_profiler = None',
    'scalene opcional'
)
replace_once(
    bin_file,
    '    sys.settrace(tracefunc)',
    '    # sys.settrace(tracefunc)  # desactivado: perfilado, no lógica del algoritmo',
    'desactivar trazado global'
)
replace_once(
    embed_file,
    '    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")',
    '    device = torch.device("cpu" if cpu_only else ("cuda" if torch.cuda.is_available() else "cpu"))',
    'respetar --cpu_only en embeddings'
)
replace_once(
    utils_file,
    'from harmony import harmonize',
    'try:\n    from harmony import harmonize\nexcept ImportError:\n    harmonize = None  # vcMSA usa pyComBat en esta réplica',
    'hacer opcional Harmony no utilizado'
)

old_faiss = '''    res = faiss.StandardGpuResources()  # use a single GPU
   #print("Cuda available?", torch.cuda.is_available())

    if torch.cuda.is_available():
       index = faiss.index_cpu_to_gpu(res, 0, index)
    else:
       logging.debug("KNN search significantly faster on a GPU")'''
new_faiss = '''    use_faiss_gpu = (
        not cpu_only
        and torch.cuda.is_available()
        and hasattr(faiss, "StandardGpuResources")
        and hasattr(faiss, "index_cpu_to_gpu")
    )
    if use_faiss_gpu:
       res = faiss.StandardGpuResources()
       index = faiss.index_cpu_to_gpu(res, 0, index)
    else:
       logging.debug("FAISS ejecutándose en CPU")'''
replace_once(bin_file, old_faiss, new_faiss, 'fallback de FAISS CPU')

patch_diff = command_output(['git', '-C', str(VCMSA_DIR), 'diff', '--no-ext-diff'])
(EVIDENCE_DIR / 'vcmsa_colab.patch').write_text(patch_diff + '\n', encoding='utf-8')

patch_notes = '''# Parches de compatibilidad

1. `scalene` pasa a ser opcional: solo es una herramienta de perfilado.
2. Se desactiva `sys.settrace`: el trazado global no cambia el alineamiento y altera fuertemente los tiempos.
3. `--cpu_only` selecciona realmente CPU durante embeddings.
4. FAISS usa CPU cuando el paquete instalado no contiene la API GPU. La GPU se mantiene para ProtT5.
5. Harmony pasa a ser opcional porque la ruta reproducida usa pyComBat, tal como el código original.

No se modifican el cálculo de embeddings, RBH, MNCM, clustering, construcción del DAG ni llenado iterativo.
'''
(EVIDENCE_DIR / 'patch_notes.md').write_text(patch_notes, encoding='utf-8')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(VCMSA_DIR)], check=True)
help_check = subprocess.run([sys.executable, str(bin_file), '--help'], text=True, capture_output=True)
if help_check.returncode != 0:
    print(help_check.stdout)
    print(help_check.stderr)
    raise RuntimeError('vcMSA no superó la comprobación de instalación.')
print('vcMSA instalado y parche documentado.')

Ya aplicado: scalene opcional
Ya aplicado: desactivar trazado global
Ya aplicado: respetar --cpu_only en embeddings
Ya aplicado: hacer opcional Harmony no utilizado
Ya aplicado: fallback de FAISS CPU
vcMSA instalado y parche documentado.


## 4. Registro del entorno

Esta celda guarda las versiones reales del runtime. Es importante porque Colab cambia con el tiempo.

In [21]:
def safe_capture(cmd):
    try:
        result = subprocess.run(cmd, text=True, capture_output=True, timeout=120)
        return (result.stdout + result.stderr).strip()
    except Exception as exc:
        return f'ERROR: {type(exc).__name__}: {exc}'

environment_sections = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'platform': platform.platform(),
    'python': safe_capture([sys.executable, '--version']),
    'uname': safe_capture(['uname', '-a']),
    'lscpu': safe_capture(['lscpu']),
    'nvidia_smi': safe_capture(['nvidia-smi']),
    'pip_freeze': safe_capture([sys.executable, '-m', 'pip', 'freeze']),
    'vcmsa_commit': vcmsa_head,
    'benchmark_commit': benchmark_head,
}
environment_text = '\n\n'.join(f'===== {key} =====\n{value}' for key, value in environment_sections.items())
(EVIDENCE_DIR / 'environment.txt').write_text(environment_text + '\n', encoding='utf-8')

config = {
    'dataset_mode': DATASET_MODE,
    'quantest2_family': QUANTEST2_FAMILY,
    'model': MODEL_NAME,
    'layers': LAYERS,
    'embedding_dimension_per_amino_acid': 16 * 1024,
    'padding': PADDING,
    'sequence_similarity_threshold': SEQSIM_THRESHOLD,
    'batch_correction': USE_BATCH_CORRECTION,
    'maximum_noncrossing': USE_MNCM,
    'maximum_increasing_subsequence': USE_MIS,
    'outlier_exclusion': USE_OUTLIER_EXCLUSION,
    'faiss_device': 'cpu',
    'embedding_device': 'cuda',
    'vcmsa_commit': VCMSA_COMMIT,
    'benchmark_commit': BENCHMARK_COMMIT,
}
(EVIDENCE_DIR / 'configuration.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
print('Entorno y configuración registrados.')

Entorno y configuración registrados.


## 5. Preparación del conjunto de prueba

Para QuanTest2 se utiliza el conjunto de 20 secuencias preparado por
los autores. Las tres secuencias con estructura conocida también
aparecen en el alineamiento de referencia y las otras 17 proporcionan
el contexto adicional utilizado durante el alineamiento.

In [22]:
REFERENCE_PATH = None

if DATASET_MODE == 'quantest2':
    reference_source = (
        BENCHMARK_DIR
        / 'quantest2_files'
        / 'Ref'
        / f'{QUANTEST2_FAMILY}.ref'
    )

    test_source = (
        BENCHMARK_DIR
        / 'quantest2_files'
        / 'Test'
        / f'{QUANTEST2_FAMILY}.vie.20seqs.fasta'
    )

    if not reference_source.exists():
        raise FileNotFoundError(
            f'No se encontró la referencia: {reference_source}'
        )

    if not test_source.exists():
        raise FileNotFoundError(
            f'No se encontró el conjunto de 20 secuencias: {test_source}'
        )

    REFERENCE_PATH = (
        EVIDENCE_DIR
        / f'{QUANTEST2_FAMILY}.reference.fasta'
    )

    INPUT_PATH = (
        EVIDENCE_DIR
        / f'{QUANTEST2_FAMILY}.input20.fasta'
    )

    shutil.copy2(reference_source, REFERENCE_PATH)
    shutil.copy2(test_source, INPUT_PATH)

elif DATASET_MODE == 'zf10':
    source = VCMSA_DIR / 'example_files' / 'zf10.fasta'
    INPUT_PATH = EVIDENCE_DIR / 'zf10.input.fasta'
    shutil.copy2(source, INPUT_PATH)

else:
    raise ValueError(
        "DATASET_MODE debe ser 'quantest2' o 'zf10'."
    )

input_records = list(SeqIO.parse(str(INPUT_PATH), 'fasta'))

if len(input_records) < 2:
    raise ValueError('Se necesitan por lo menos dos secuencias.')

if len({record.id for record in input_records}) != len(input_records):
    raise ValueError('Los identificadores FASTA deben ser únicos.')

if DATASET_MODE == 'quantest2':
    if len(input_records) != 20:
        raise ValueError(
            f'Se esperaban 20 secuencias, pero se encontraron {len(input_records)}.'
        )

    reference_records = list(
        SeqIO.parse(str(REFERENCE_PATH), 'fasta')
    )

    input_by_id = {
        record.id: str(record.seq).upper()
        for record in input_records
    }

    reference_by_id = {
        record.id: str(record.seq)
        .replace('-', '')
        .replace('.', '')
        .upper()
        for record in reference_records
    }

    missing_reference_ids = (
        set(reference_by_id) - set(input_by_id)
    )

    if missing_reference_ids:
        raise ValueError(
            'Las secuencias de referencia no aparecen en la entrada: '
            f'{sorted(missing_reference_ids)}'
        )

    different_sequences = [
        sequence_id
        for sequence_id in reference_by_id
        if reference_by_id[sequence_id] != input_by_id[sequence_id]
    ]

    if different_sequences:
        raise ValueError(
            'Las secuencias de entrada no coinciden con la referencia: '
            f'{different_sequences}'
        )

    reference_ids = set(reference_by_id)

else:
    reference_records = []
    reference_ids = set()

dataset_summary = pd.DataFrame({
    'id': [record.id for record in input_records],
    'longitud_sin_gaps': [
        len(record.seq) for record in input_records
    ],
    'es_secuencia_de_referencia': [
        record.id in reference_ids for record in input_records
    ],
})

dataset_summary.to_csv(
    EVIDENCE_DIR / 'dataset_summary.csv',
    index=False
)

print(f'Secuencias de entrada: {len(input_records)}')

if reference_records:
    print(f'Secuencias estructurales de referencia: {len(reference_records)}')
    print(
        'Secuencias adicionales: '
        f'{len(input_records) - len(reference_records)}'
    )

print(
    'Longitud mínima/media/máxima: '
    f'{dataset_summary.longitud_sin_gaps.min()} / '
    f'{dataset_summary.longitud_sin_gaps.mean():.1f} / '
    f'{dataset_summary.longitud_sin_gaps.max()}'
)

display(dataset_summary)

Secuencias de entrada: 20
Secuencias estructurales de referencia: 3
Secuencias adicionales: 17
Longitud mínima/media/máxima: 61 / 65.5 / 69


,id,longitud_sin_gaps,es_secuencia_de_referencia
0,seq0001,66,True
1,seq0002,67,True
2,seq0003,69,True
3,seq0092,65,False
4,seq0129,65,False
5,seq0130,66,False
6,seq0174,65,False
7,seq0208,67,False
8,seq0232,64,False
9,seq0282,66,False


## 6. Ejecutor con medición de tiempo, RAM y VRAM

El embedding y el alineamiento se ejecutan por separado. Así se puede comparar con los tiempos informados en el artículo sin confundir descarga/carga del modelo con la fase algorítmica posterior.

In [23]:
TIMINGS = []
COMMANDS_LOG = EVIDENCE_DIR / 'commands.log'

def process_tree_rss_mb(pid):
    try:
        root = psutil.Process(pid)
        processes = [root] + root.children(recursive=True)
        return sum(p.memory_info().rss for p in processes if p.is_running()) / (1024 ** 2)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        return 0.0

def gpu_memory_mb_for_pid_tree(pid):
    try:
        root = psutil.Process(pid)
        pids = {root.pid, *(child.pid for child in root.children(recursive=True))}
        result = subprocess.run(
            ['nvidia-smi', '--query-compute-apps=pid,used_gpu_memory', '--format=csv,noheader,nounits'],
            text=True, capture_output=True, timeout=5
        )
        used = 0.0
        for line in result.stdout.splitlines():
            parts = [part.strip() for part in line.split(',')]
            if len(parts) == 2 and int(parts[0]) in pids:
                used += float(parts[1])
        return used
    except Exception:
        return 0.0

def total_gpu_memory_mb():
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
            text=True, capture_output=True, timeout=5
        )
        return sum(float(line.strip()) for line in result.stdout.splitlines() if line.strip())
    except Exception:
        return 0.0

def run_monitored(stage, cmd, cwd=None):
    cmd_text = shlex.join(str(part) for part in cmd)
    with COMMANDS_LOG.open('a', encoding='utf-8') as handle:
        handle.write(f'[{stage}] {cmd_text}\n')
    log_path = EVIDENCE_DIR / f'{stage}.log'
    samples_path = EVIDENCE_DIR / f'{stage}_resources.csv'
    samples = []
    start_wall = time.perf_counter()
    start_utc = datetime.now(timezone.utc).isoformat()
    with log_path.open('w', encoding='utf-8') as log_handle:
        process = subprocess.Popen(
            [str(part) for part in cmd], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1
        )
        def reader():
            assert process.stdout is not None
            for line in process.stdout:
                log_handle.write(line)
                log_handle.flush()
                print(line, end='')
        thread = threading.Thread(target=reader, daemon=True)
        thread.start()
        while process.poll() is None:
            elapsed = time.perf_counter() - start_wall
            samples.append({
                'elapsed_seconds': round(elapsed, 3),
                'ram_process_tree_mb': round(process_tree_rss_mb(process.pid), 3),
                'vram_process_tree_mb': round(gpu_memory_mb_for_pid_tree(process.pid), 3),
                'vram_total_device_mb': round(total_gpu_memory_mb(), 3),
            })
            time.sleep(1.0)
        thread.join(timeout=30)
        return_code = process.wait()
    elapsed = time.perf_counter() - start_wall
    pd.DataFrame(samples).to_csv(samples_path, index=False)
    row = {
        'stage': stage,
        'start_utc': start_utc,
        'elapsed_seconds': round(elapsed, 3),
        'peak_ram_process_tree_mb': max((s['ram_process_tree_mb'] for s in samples), default=0.0),
        'peak_vram_process_tree_mb': max((s['vram_process_tree_mb'] for s in samples), default=0.0),
        'peak_vram_total_device_mb': max((s['vram_total_device_mb'] for s in samples), default=0.0),
        'return_code': return_code,
        'command': cmd_text,
    }
    TIMINGS.append(row)
    pd.DataFrame(TIMINGS).to_csv(EVIDENCE_DIR / 'timings.csv', index=False)
    if return_code != 0:
        tail = '\n'.join(log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-40:])
        raise RuntimeError(f'Falló {stage} con código {return_code}. Últimas líneas:\n{tail}')
    return row

print('Monitor preparado.')

Monitor preparado.


## 7. Generación de embeddings con ProtT5

Esta es la fase que usa la GPU. Se guarda el pickle fuera del ZIP por defecto para evitar que el archivo descargable sea excesivamente grande; su tamaño y SHA-256 sí quedan registrados.

In [24]:
VCMSA_SCRIPT = VCMSA_DIR / 'bin' / 'vcmsa'
EMBEDDINGS_PATH = CACHE_DIR / f'{INPUT_PATH.stem}.embeddings.pkl'
EMBED_PLACEHOLDER = EVIDENCE_DIR / 'embed_stage_placeholder.aln'

embedding_cmd = [
    sys.executable, str(VCMSA_SCRIPT),
    '-i', str(INPUT_PATH),
    '-o', str(EMBED_PLACEHOLDER),
    '-m', MODEL_NAME,
    '-l', *[str(layer) for layer in LAYERS],
    '-p', str(PADDING),
    '-st', str(SEQSIM_THRESHOLD),
    '-eo', '-eout', str(EMBEDDINGS_PATH),
]
embedding_result = run_monitored('01_embeddings', embedding_cmd, cwd=VCMSA_DIR)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

embedding_manifest = {
    'path_in_colab': str(EMBEDDINGS_PATH),
    'size_bytes': EMBEDDINGS_PATH.stat().st_size,
    'sha256': sha256_file(EMBEDDINGS_PATH),
    'included_in_zip': INCLUDE_EMBEDDINGS_IN_ZIP,
}
(EVIDENCE_DIR / 'embedding_manifest.json').write_text(json.dumps(embedding_manifest, indent=2), encoding='utf-8')
print(json.dumps(embedding_manifest, indent=2))

2026-07-22 14:35:25.456937: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
CUDA available? True
This is a t5 model
load tokenizer
load_model:model_path Rostlab/prot_t5_xl_uniref50
T5 model
tokenizer loaded
model_loaded
This is a t5 model
Mode

## 8. Alineamiento vcMSA

Se reutilizan los embeddings guardados. FAISS se ejecuta en CPU porque el paquete `faiss-gpu` original no dispone de ruedas compatibles con los runtimes modernos de Colab; ProtT5 sí fue ejecutado en GPU.

In [25]:
VCMSA_OUTPUT = EVIDENCE_DIR / f'vcmsa_{INPUT_PATH.stem}.clustal.aln'
alignment_cmd = [
    sys.executable, str(VCMSA_SCRIPT),
    '-i', str(INPUT_PATH),
    '-e', str(EMBEDDINGS_PATH),
    '-o', str(VCMSA_OUTPUT),
    '-m', MODEL_NAME,
    '-l', *[str(layer) for layer in LAYERS],
    '-p', str(PADDING),
    '-st', str(SEQSIM_THRESHOLD),
    '-co', '-dc',
]
if USE_BATCH_CORRECTION:
    alignment_cmd.append('-bc')
if USE_OUTLIER_EXCLUSION:
    alignment_cmd.append('-ex')
if USE_MNCM:
    alignment_cmd.append('-mnc')
if USE_MIS:
    alignment_cmd.append('-mis')

alignment_result = run_monitored('02_vcmsa_alignment', alignment_cmd, cwd=VCMSA_DIR)
print('Alineamiento:', VCMSA_OUTPUT)

2026-07-22 14:36:13.550473: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Found 20 batches.
Adjusting for 0 covariate(s) or covariate level(s).
Standardizing Data across genes.
Fitting L/S model and finding priors.
Finding parametric adjustments.
Adjusting the Data
Alineamiento: /content/vcmsa_paper_replica/runs/20260722T143442Z/vcmsa_csp.input20.clustal.aln


## 9. Validaciones automáticas

Se comprueba que no falte ninguna secuencia, que todas las filas tengan la misma longitud y que vcMSA no haya perdido, agregado ni sustituido residuos.

In [26]:
def read_alignment(path, fmt):
    return AlignIO.read(str(path), fmt)

def validate_alignment(input_fasta, alignment_path, alignment_format):
    inputs = {record.id: str(record.seq).upper() for record in SeqIO.parse(str(input_fasta), 'fasta')}
    alignment = read_alignment(alignment_path, alignment_format)
    outputs = {record.id: str(record.seq).replace('-', '').replace('.', '').upper() for record in alignment}
    result = {
        'input_sequence_count': len(inputs),
        'output_sequence_count': len(outputs),
        'same_identifiers': set(inputs) == set(outputs),
        'uniform_alignment_width': len({len(record.seq) for record in alignment}) == 1,
        'alignment_width': alignment.get_alignment_length(),
        'residues_preserved': set(inputs) == set(outputs) and all(inputs[key] == outputs[key] for key in inputs),
        'different_or_missing_ids': sorted(set(inputs) ^ set(outputs)),
    }
    return result, alignment

vcmsa_validation, vcmsa_alignment = validate_alignment(INPUT_PATH, VCMSA_OUTPUT, 'clustal')
(EVIDENCE_DIR / 'vcmsa_validation.json').write_text(json.dumps(vcmsa_validation, indent=2), encoding='utf-8')
print(json.dumps(vcmsa_validation, indent=2))
if not all([vcmsa_validation['same_identifiers'], vcmsa_validation['uniform_alignment_width'], vcmsa_validation['residues_preserved']]):
    raise AssertionError('El alineamiento no superó la validación de integridad; revise vcmsa_validation.json.')

print('\nVista parcial del alineamiento:')
for record in vcmsa_alignment[:5]:
    print(f'{record.id:18s} {str(record.seq)[:120]}')

{
  "input_sequence_count": 20,
  "output_sequence_count": 20,
  "same_identifiers": true,
  "uniform_alignment_width": true,
  "alignment_width": 72,
  "residues_preserved": true,
  "different_or_missing_ids": []
}

Vista parcial del alineamiento:
seq0001            --MQRGKVKWFNNEKGYGFIEVE-GGSDVFVHFTAIQ-GEGFKTLEEGQEVSFEIVQGNRG-PQAANVVKL-
seq0002            --MLEGKVKWFNSEKGFGFIEVE-GQDDVFVHFSAIQ-GEGFKTLEEGQAVSFEIVEGNRG-PQAANVTKEA
seq0003            SGKMTGIVKWFNADKGFGFITPDDGSKDVFVHFSAIQ-NDGYKSLDEGQKVSFTIESGAKG-PAAGNVTSL-
seq0092            ---QTGKVKWFNGEKGFGFIEVE-GGEDVFVHFSAIQ-GDGFKTLEEGQEVSFEIVDGNRG-PQAANVTKN-
seq0129            ---VTGTVKWFNETKGFGFISQDNGGKDVFVHFSSIV-SEGFKTLAEGQKVTFTVEEGQKG-PQASNVVV--


## 10. Ejecución de Código Propio
El código propio de compone de 3 partes, primero las librerías y la función para leer fasta

In [27]:
from numba import njit
from Bio import SeqIO

def leer_fasta_multi(ruta):
    records = []
    current_id = None
    current_seq = []
    with open(ruta, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id:
                    records.append((current_id, "".join(current_seq)))
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id:
            records.append((current_id, "".join(current_seq)))
    return records

Funciones para alineamiento normal

In [28]:
def alinear_cadenas(cadena1, cadena2, posicion, tipo, alineacion1, alineacion2):
    if(tipo == 0):
        alineacion1 = cadena1[posicion[0] - 1] + alineacion1
        alineacion2 = cadena2[posicion[1] - 1] + alineacion2
    elif(tipo == 1):
        alineacion1 = cadena1[posicion[0] - 1] + alineacion1
        alineacion2 = "-" + alineacion2
    else:
        alineacion2 = cadena2[posicion[1] - 1] + alineacion2
        alineacion1 = "-" + alineacion1
    return alineacion1, alineacion2
@njit
def llenar_matriz_global(cadena1, cadena2, match, mismatch, gap):
    m = len(cadena1) + 1
    n = len(cadena2) + 1
    matriz = np.zeros((m, n), dtype=np.int32)
    for i in range(m):
        matriz[i, 0] = gap * i
    for j in range(n):
        matriz[0, j] = gap * j
    for i in range(1, m):
        for j in range(1, n):
            diagonal = matriz[i - 1, j - 1] + (match if (cadena1[i - 1] == cadena2[j - 1] or
                                                        cadena1[i - 1] == 'X' or
                                                        cadena2[j - 1] == 'X') else mismatch)
            arriba = matriz[i - 1, j] + gap
            izquierda = matriz[i, j - 1] + gap
            matriz[i, j] = max(diagonal, arriba, izquierda)
    return matriz
@njit
def llenar_matriz_local(cadena1, cadena2, match, mismatch, gap):
    m = len(cadena1) + 1
    n = len(cadena2) + 1
    matriz = np.zeros((m, n), dtype=np.int32)
    puntaje_maximo = 0
    posicion_maxima = (0, 0)
    for i in range(1, m):
        for j in range(1, n):
            diagonal = matriz[i - 1, j - 1] + (match if (cadena1[i - 1] == cadena2[j - 1] or
                                                        cadena1[i - 1] == 'X' or
                                                        cadena2[j - 1] == 'X') else mismatch)
            arriba = matriz[i - 1, j] + gap
            izquierda = matriz[i, j - 1] + gap
            matriz[i, j] = max(0, diagonal, arriba, izquierda)
            if matriz[i, j] > puntaje_maximo:
                puntaje_maximo = matriz[i, j]
                posicion_maxima = (i, j)
    return matriz, posicion_maxima
def disponibles(matriz, i, j, cadena1, cadena2, match, mismatch, gap):
    diag = False
    if i > 0 and j > 0:
        diagonal = matriz[i - 1][j - 1] + (match if (cadena1[i - 1] == cadena2[j - 1] or
                                                    cadena1[i - 1] == 'X' or
                                                    cadena2[j - 1] == 'X') else mismatch)
        diag = (matriz[i][j] == diagonal)
    arri = False
    if i > 0:
        arriba = matriz[i - 1][j] + gap
        arri = (matriz[i][j] == arriba)
    izqu = False
    if j > 0:
        izquierda = matriz[i][j - 1] + gap
        izqu = (matriz[i][j] == izquierda)
    return diag, arri, izqu
def ajustar_salidas(condiciones, alin01, alin02, resultado, tiempo, matriz, pos_max, pos_i, pos_j):
    con_puntaje = condiciones['puntaje']
    con_tiempo = condiciones['tiempo']
    con_matriz = condiciones['matriz']
    con_posiciones = condiciones['posiciones']
    con_alineamiento = condiciones['alineamiento']
    if(con_puntaje):
        resultado.append(matriz[pos_max])
    if(con_tiempo):
        resultado.append(tiempo)
    if(con_alineamiento):
        resultado.append(alin01)
        resultado.append(alin02)
    if(con_matriz):
        resultado.append(matriz)
    if(con_posiciones):
        resultado.append((pos_i, pos_j))
def condicion_allocal(matriz, i, j):
    return matriz[i][j] != 0
def condicion_alglobal(matriz, i, j):
    return i > 0 or j > 0
def back_tracking(matriz, posicion, cadena1, cadena2, parametros, condiciones, tipo):
    match = parametros['match']
    mismatch = parametros['mismatch']
    gap = parametros['gap']
    con_posiciones = condiciones['posiciones']
    con_alineamiento = condiciones['alineamiento']
    alineacion1 = ""
    alineacion2 = ""
    i, j = posicion
    pos_i = []
    pos_j = []
    if tipo == 0:
        condicion = condicion_allocal
    else:
        condicion = condicion_alglobal
    while condicion(matriz, i, j):
        if(con_posiciones):
            pos_i.append(i-1)
            pos_j.append(j-1)
        di_score, up_score, le_score = disponibles(matriz, i, j, cadena1, cadena2, match, mismatch, gap)
        tipo_alin = 0
        n_i = i
        n_j = j
        if di_score:
            tipo_alin = 0
            n_i = n_i-1
            n_j = n_j-1
        elif up_score:
            tipo_alin = 1
            n_i = n_i-1
        elif le_score:
            tipo_alin = 2
            n_j = n_j-1
        if(con_alineamiento):
            alineacion1, alineacion2 = alinear_cadenas(cadena1, cadena2, (i, j), tipo_alin, alineacion1, alineacion2)
        i = n_i
        j = n_j
    return alineacion1, alineacion2, pos_i, pos_j
def alineamiento_local(cadena1, cadena2, parametros, condiciones):
    match = parametros['match']
    mismatch = parametros['mismatch']
    gap = parametros['gap']
    con_tiempo = condiciones['tiempo']
    con_posiciones = condiciones['posiciones']
    con_alineamiento = condiciones['alineamiento']
    if(con_tiempo):
        inicio = time.perf_counter()
    matriz, posicion_maxima = llenar_matriz_local(cadena1, cadena2, match, mismatch, gap)
    if(con_alineamiento or con_posiciones):
        alineacion1, alineacion2, pos_i, pos_j = back_tracking(matriz, posicion_maxima, cadena1, cadena2,
                                                           parametros, con_posiciones, 0)
    if(con_tiempo):
        final = time.perf_counter()
    resultado = []
    ajustar_salidas(condiciones, alineacion1, alineacion2, resultado, final - inicio, matriz, posicion_maxima, pos_i, pos_j)
    return resultado
def puntaje_global(cadena1, cadena2, parametros):
    match = parametros['match']
    mismatch = parametros['mismatch']
    gap = parametros['gap']
    matriz = llenar_matriz_global(cadena1, cadena2, match, mismatch, gap)
    posicion_maxima = (matriz.shape[0] - 1, matriz.shape[1] - 1)
    return matriz[posicion_maxima]
def alineamiento_global(cadena1, cadena2,
                        parametros = {'match':1, 'mismatch':-1, 'gap':-1},
                        condiciones = {'puntaje':False, 'tiempo':False, 'alineamiento':True,
                                       'matriz':False, 'posiciones':False}):
    match = parametros['match']
    mismatch = parametros['mismatch']
    gap = parametros['gap']
    con_tiempo = condiciones['tiempo']
    con_posiciones = condiciones['posiciones']
    con_alineamiento = condiciones['alineamiento']
    inicio = 0.0
    final = 0.0
    if(con_tiempo):
        inicio = time.perf_counter()
    matriz = llenar_matriz_global(cadena1, cadena2, match, mismatch, gap)
    posicion_maxima = (matriz.shape[0] - 1, matriz.shape[1] - 1)
    if(con_alineamiento or con_posiciones):
        alineacion1, alineacion2, pos_i, pos_j = back_tracking(matriz, posicion_maxima, cadena1, cadena2,
                                                           parametros, condiciones, 1)
    if(con_tiempo):
        final = time.perf_counter()
    resultado = []
    ajustar_salidas(condiciones, alineacion1, alineacion2, resultado, final - inicio, matriz, posicion_maxima, pos_i, pos_j)
    return resultado
def distancia(alin01, alin02, n):
    dist = 0
    for i in range(0, n):
        if((alin01[i] != '-' and alin01[i] != 'X') and
           (alin02[i] != '-' and alin02[i] != 'X') and
           (alin01[i] != alin02[i])):
            dist += 1
    return dist
def matriz_distancia(cadenas, n, parametros):
    matriz = np.zeros((n, n), dtype=np.int32)
    for i in range(0, n):
        for j in range(i+1, n):
            alin01, alin02 = alineamiento_global(cadenas[i], cadenas[j], parametros)
            matriz[i,j] = distancia(alin01, alin02, len(alin01))
            matriz[j,i] = matriz[i,j]
    return matriz
def cadenas_minimo(matriz):
    if(np.all(matriz == 0)):
        return 0 , 1
    min_no_cero = matriz[matriz != 0].min()
    x, y = np.argwhere(matriz == min_no_cero)[0]
    return x, y

Funciones para alineamiento multiple

In [29]:
def imprimir_guardar_zona(inicio, fin, cadenas, i, salto, m,
                          m_valores, funcion = lambda s: print(s, end=""), index = None):
    nombre_cadena = ""
    valor_cadena = ""
    if(index == None):
        nombre_cadena = f"Cadena {i}"
    else:
        nombre_cadena = f"Cadena {index[i]}"
    if(m_valores[i] < fin):
        if(inicio < m_valores[i]):
            extras = m - m_valores[i]
            espacio = ' ' * extras
            valor_cadena = f"{cadenas[i][inicio:m_valores[i]]}{espacio}"
        else:
            espacio = ' ' * salto
            valor_cadena = f"{espacio}"
    else:
        valor_cadena = f"{cadenas[i][inicio:fin]}"
    funcion(f"{nombre_cadena}: 	{valor_cadena}\n")

def imprimir_guardar_cadenas(cadenas, max_ancho = 100, dif = 16,
                             funcion = lambda s: print(s, end=""), index = None):
    n = len(cadenas)
    m_valores = []
    m = 0
    for i in range(n):
        m = max(m, len(cadenas[i]))
        m_valores.append(len(cadenas[i]))
    inicio = 0
    salto = max_ancho - dif
    fin = salto
    while(fin < m):
        for i in range(n):
            imprimir_guardar_zona(inicio, fin, cadenas, i, salto, m, m_valores, funcion, index)
        funcion("\n")
        inicio += salto
        fin += salto
    for i in range(n):
        imprimir_guardar_zona(inicio, m, cadenas, i, salto, m, m_valores, funcion, index)

def cadenas_minimo(matriz):
    if(np.all(matriz == 0)):
        return 0 , 1
    min_no_cero = matriz[matriz != 0].min()
    x, y = np.argwhere(matriz == min_no_cero)[0]
    return x, y

def insertar_gaps(cadena, gaps):
    res = []
    gaps_hechos = 0
    inicio = 0
    for gap in gaps:
        res.append(cadena[inicio:(gap+gaps_hechos)])
        res.append('-')
        inicio = (gap+gaps_hechos)
        gaps_hechos -= 1
    res.append(cadena[inicio:])
    return ''.join(res)

def ajustar_gaps(cadenas, var_i, var_j, min_i, min_j, n, m):
    pos_min_i = [i for i, c in enumerate(cadenas[var_i[min_i]]) if c == '-']
    pos_min_j = [j for j, c in enumerate(cadenas[var_j[min_j]]) if c == '-']
    for i in range(n):
        if(i != min_i):
            cadenas[var_i[i]] = insertar_gaps(cadenas[var_i[i]], pos_min_i)
        cadenas[var_i[i]] = cadenas[var_i[i]].replace('-', 'X')
    for j in range(m):
        if(j != min_j):
            cadenas[var_j[j]] = insertar_gaps(cadenas[var_j[j]], pos_min_j)
        cadenas[var_j[j]] = cadenas[var_j[j]].replace('-', 'X')

def alineamiento_minimo(cadenas, var_i, var_j, parametros, n, m):
    matriz = np.zeros((n, m), dtype=np.int32)
    for i in range(0, n):
        for j in range(0, m):
            alin_var0i, alin_var1j = alineamiento_global(cadenas[var_i[i]], cadenas[var_j[j]], parametros)
            matriz[i,j] = distancia(alin_var0i, alin_var1j, len(alin_var0i))
    return cadenas_minimo(matriz)

def arbol_guia(matriz, n, max_ancho = 80, imprimir = False):
    tmp_n = n
    tmp_matriz = matriz.copy().astype(np.float64)
    arbol = []
    for i in range(n):
        arbol.append(i)
    if(imprimir):
        print("=" * max_ancho)
        print("Arbol inicial:\n",arbol)
        print("Matriz inicial:\n",tmp_matriz)
    while(tmp_n > 2):
        i, j = cadenas_minimo(tmp_matriz)
        arbol.append([arbol[i],arbol[j]])
        arbol.remove(arbol[j])
        arbol.remove(arbol[i])
        nuevos_valores = []
        for k in range(tmp_n):
            if(k != i and  k != j):
                nuevos_valores.append((tmp_matriz[k, i] + tmp_matriz[k, j])/2.0)
        tmp_matriz = np.delete(np.delete(tmp_matriz, [i, j], axis=0), [i, j], axis=1)
        tmp_matriz = np.vstack((tmp_matriz, nuevos_valores))
        nuevos_valores.append(0.0)
        tmp_matriz = np.column_stack((tmp_matriz, nuevos_valores))
        tmp_n -= 1
        if(imprimir and tmp_n > 2):
            print("-" * max_ancho)
            print("Arbol actual:\n",arbol)
            print("Matriz actual:\n",tmp_matriz)
    if(imprimir):
        print("-" * max_ancho)
        print("Arbol final:\n",arbol)
    return arbol

def recorrer_arbol(data0, data1, cadenas, parametros,
                   impresion_config = (80, False)):
    var_i = data0
    var_j = data1
    if(type(data0) is list):
        var_i = recorrer_arbol(data0[0], data0[1], cadenas, parametros, impresion_config)
    if(type(data1) is list):
        var_j = recorrer_arbol(data1[0], data1[1], cadenas, parametros, impresion_config)
    if(type(var_i) is int and type(var_j) is int):
        cadenas[var_i], cadenas[var_j] = alineamiento_global(cadenas[var_i], cadenas[var_j], parametros)
        cadenas[var_i] = cadenas[var_i].replace('-', 'X')
        cadenas[var_j] = cadenas[var_j].replace('-', 'X')
        if(impresion_config[1]):
            print("-" * impresion_config[0])
            imprimir_guardar_cadenas(cadenas, impresion_config[0])
        return [var_i, var_j]
    else:
        if(type(var_i) is int):
            var_i = [var_i]
        if(type(var_j) is int):
            var_j = [var_j]
        n = len(var_i)
        m = len(var_j)
        min_i, min_j = alineamiento_minimo(cadenas, var_i, var_j, parametros, n, m)
        cadenas[var_i[min_i]], cadenas[var_j[min_j]] = alineamiento_global(cadenas[var_i[min_i]], cadenas[var_j[min_j]], parametros)
        ajustar_gaps(cadenas, var_i, var_j, min_i, min_j, n, m)
        if(impresion_config[1]):
            print("-" * impresion_config[0])
            imprimir_guardar_cadenas(cadenas, impresion_config[0])
        return var_i + var_j

def revertir_x(cadenas):
    for i in range(len(cadenas)):
        cadenas[i] = cadenas[i].replace('X', '-')

def alineamiento_multiple(cadenas, max_ancho=80, n=None,
                          parametros = {'match':1, 'mismatch':-1, 'gap':-1},
                          imprimir = False):
    if n is None:
        n = len(cadenas)
    cadenas_alineadas = cadenas.copy()
    matriz = matriz_distancia(cadenas_alineadas, n, parametros)
    if(imprimir):
        print("=" * max_ancho)
        print("Matriz de distancia:\n",matriz)
    arbol = arbol_guia(matriz, n, max_ancho, imprimir)
    if(imprimir):
        print("=" * max_ancho)
        print("Fusión de Secuencias:")
    recorrer_arbol(arbol[0], arbol[1],cadenas_alineadas,
                   parametros, (max_ancho, imprimir))
    revertir_x(cadenas_alineadas)
    return cadenas_alineadas

In [30]:
MY_ALIGNMENT_PATH = EVIDENCE_DIR / f'msa_propio_{INPUT_PATH.stem}.fasta'

print('Ejecutando Cdigo Propio de alineamiento mltiple...')
start_propio = time.perf_counter()

input_records_propio = list(SeqIO.parse(str(INPUT_PATH), 'fasta'))
ids_propio = [rec.id for rec in input_records_propio]
seqs_propio = [str(rec.seq).upper() for rec in input_records_propio]

aligned_seqs_propio = alineamiento_multiple(seqs_propio)
elapsed_propio = time.perf_counter() - start_propio

propio_seq_records = [
    SeqRecord(Seq(s), id=i, description='')
    for i, s in zip(ids_propio, aligned_seqs_propio)
]
SeqIO.write(propio_seq_records, str(MY_ALIGNMENT_PATH), 'fasta')

shutil.copy2(MY_ALIGNMENT_PATH, 'msa_propio.fasta')

TIMINGS.append({
    'stage': '04_codigo_propio',
    'start_utc': datetime.now(timezone.utc).isoformat(),
    'elapsed_seconds': round(elapsed_propio, 3),
    'peak_ram_process_tree_mb': None,
    'peak_vram_process_tree_mb': None,
    'peak_vram_total_device_mb': None,
    'return_code': 0,
    'command': 'alineamiento_multiple (Python / Numba)',
})
pd.DataFrame(TIMINGS).to_csv(EVIDENCE_DIR / 'timings.csv', index=False)

my_validation, my_alignment = validate_alignment(INPUT_PATH, MY_ALIGNMENT_PATH, 'fasta')
(EVIDENCE_DIR / 'propio_validation.json').write_text(json.dumps(my_validation, indent=2), encoding='utf-8')

print(f'Alineamiento propio generado con xito en {elapsed_propio:.3f} s:')
print(f'  Archivo: {MY_ALIGNMENT_PATH}')
print(f'  Ancho del alineamiento: {my_validation["alignment_width"]}')
print(f'  Residuos preservados: {my_validation["residues_preserved"]}')

Ejecutando Cdigo Propio de alineamiento mltiple...
Alineamiento propio generado con xito en 0.934 s:
  Archivo: /content/vcmsa_paper_replica/runs/20260722T143442Z/msa_propio_csp.input20.fasta
  Ancho del alineamiento: 85
  Residuos preservados: True


## 11. Total Column Score y comparación entre métodos

Para calcular el TCS se extraen del alineamiento producido las tres
secuencias que aparecen en la referencia estructural. Se eliminan las
columnas que quedan completamente vacías y luego se comparan las
columnas resultantes con la referencia.

In [31]:
from Bio.Align import MultipleSeqAlignment

def project_alignment(alignment, ordered_ids):
    records = {
        record.id: str(record.seq)
        for record in alignment
    }

    missing = set(ordered_ids) - set(records)

    if missing:
        raise ValueError(
            'Faltan identificadores de referencia en el alineamiento: '
            f'{sorted(missing)}'
        )

    width = alignment.get_alignment_length()

    columns_to_keep = [
        column
        for column in range(width)
        if any(
            records[sequence_id][column] not in '-.'
            for sequence_id in ordered_ids
        )
    ]

    projected_records = []

    for sequence_id in ordered_ids:
        projected_sequence = ''.join(
            records[sequence_id][column]
            for column in columns_to_keep
        )

        projected_records.append(
            SeqRecord(
                Seq(projected_sequence),
                id=sequence_id,
                description=''
            )
        )

    return MultipleSeqAlignment(projected_records)


def column_signatures(alignment, ordered_ids):
    records = {
        record.id: str(record.seq)
        for record in alignment
    }

    counters = {
        sequence_id: 0
        for sequence_id in ordered_ids
    }

    signatures = []
    width = alignment.get_alignment_length()

    for column in range(width):
        signature = []
        any_residue = False

        for sequence_id in ordered_ids:
            amino_acid = records[sequence_id][column]

            if amino_acid in '-.':
                signature.append(None)
            else:
                counters[sequence_id] += 1
                signature.append(counters[sequence_id])
                any_residue = True

        if any_residue:
            signatures.append(tuple(signature))

    return signatures


def total_column_score(reference_alignment, predicted_alignment):
    ordered_ids = [
        record.id for record in reference_alignment
    ]

    projected_alignment = project_alignment(
        predicted_alignment,
        ordered_ids
    )

    reference_columns = column_signatures(
        reference_alignment,
        ordered_ids
    )

    predicted_columns = column_signatures(
        projected_alignment,
        ordered_ids
    )

    reference_set = set(reference_columns)
    predicted_set = set(predicted_columns)

    correct_output_columns = sum(
        signature in reference_set
        for signature in predicted_columns
    )

    recovered_reference_columns = sum(
        signature in predicted_set
        for signature in reference_columns
    )

    metrics = {
        'correct_output_columns': correct_output_columns,
        'output_columns_evaluated': len(predicted_columns),
        'reference_columns': len(reference_columns),

        'tcs_fraction': (
            correct_output_columns / len(predicted_columns)
            if predicted_columns else float('nan')
        ),

        'tcs_percent': (
            100 * correct_output_columns / len(predicted_columns)
            if predicted_columns else float('nan')
        ),

        'reference_column_recall_percent': (
            100 * recovered_reference_columns / len(reference_columns)
            if reference_columns else float('nan')
        ),
    }

    return metrics, projected_alignment


metrics_rows = []

if REFERENCE_PATH is not None:
    reference_alignment = read_alignment(
        REFERENCE_PATH,
        'fasta'
    )

    ###### 1 vcMSA
    vcmsa_tcs, vcmsa_projected = total_column_score(
        reference_alignment,
        vcmsa_alignment
    )

    VCMSA_PROJECTED_PATH = (
        EVIDENCE_DIR
        / f'vcmsa_{QUANTEST2_FAMILY}.reference_sequences.fasta'
    )

    AlignIO.write(
        vcmsa_projected,
        str(VCMSA_PROJECTED_PATH),
        'fasta'
    )

    metrics_rows.append({
        'method': 'vcMSA',
        **vcmsa_tcs,
        **vcmsa_validation
    })

    ###### 2 Código Propio
    if 'MY_ALIGNMENT_PATH' in globals() and MY_ALIGNMENT_PATH.exists():
        my_tcs, my_projected = total_column_score(
            reference_alignment,
            my_alignment
        )

        MY_PROJECTED_PATH = (
            EVIDENCE_DIR
            / f'propio_{QUANTEST2_FAMILY}.reference_sequences.fasta'
        )

        AlignIO.write(
            my_projected,
            str(MY_PROJECTED_PATH),
            'fasta'
        )

        metrics_rows.append({
            "method": "MSA propio",
            **my_tcs,
            **my_validation
        })

else:
    print(
        'zf10 no tiene referencia estructural; '
        'no se calcula TCS.'
    )


metrics_df = pd.DataFrame(metrics_rows)

metrics_df.to_csv(
    EVIDENCE_DIR / 'metrics.csv',
    index=False
)

if not metrics_df.empty:
    display(metrics_df[[
        'method',
        'tcs_percent',
        'correct_output_columns',
        'output_columns_evaluated',
        'reference_columns',
        'reference_column_recall_percent',
        'alignment_width',
        'residues_preserved'
    ]])
else:
    display(metrics_df)

,method,tcs_percent,correct_output_columns,output_columns_evaluated,reference_columns,reference_column_recall_percent,alignment_width,residues_preserved
0,vcMSA,100.000000,70,70,70,100.000000,72,True
1,MSA propio,76.315789,58,76,70,82.857143,85,True


## 12. Resumen y paquete de evidencias

El ZIP excluye los embeddings por defecto, pero conserva su hash. Para descargar también el pickle, cambie `INCLUDE_EMBEDDINGS_IN_ZIP` a `True` antes de ejecutar esta celda.

In [32]:
summary = {
    'run_id': RUN_ID,
    'completed_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': DATASET_MODE if DATASET_MODE == 'zf10' else f'QuanTest2/{QUANTEST2_FAMILY}',
    'input_sha256': sha256_file(INPUT_PATH),
    'reference_sha256': sha256_file(REFERENCE_PATH) if REFERENCE_PATH else None,
    'vcmsa_output_sha256': sha256_file(VCMSA_OUTPUT),
    'propio_output_sha256': sha256_file(MY_ALIGNMENT_PATH) if 'MY_ALIGNMENT_PATH' in globals() and MY_ALIGNMENT_PATH.exists() else None,
    'vcmsa_validation': vcmsa_validation,
    'metrics': metrics_rows,
    'timings': TIMINGS,
}
(EVIDENCE_DIR / 'run_summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')

if INCLUDE_EMBEDDINGS_IN_ZIP:
    shutil.copy2(EMBEDDINGS_PATH, EVIDENCE_DIR / EMBEDDINGS_PATH.name)

manifest_lines = []
for path in sorted(p for p in EVIDENCE_DIR.rglob('*') if p.is_file() and p.name != 'manifest_sha256.txt'):
    manifest_lines.append(f'{sha256_file(path)}  {path.relative_to(EVIDENCE_DIR)}')
(EVIDENCE_DIR / 'manifest_sha256.txt').write_text('\n'.join(manifest_lines) + '\n', encoding='utf-8')

ZIP_PATH = BASE_DIR / f'vcmsa_replica_evidencias_{RUN_ID}.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(p for p in EVIDENCE_DIR.rglob('*') if p.is_file()):
        archive.write(path, arcname=path.relative_to(EVIDENCE_DIR))

print('ZIP:', ZIP_PATH)
print(f'Tamaño: {ZIP_PATH.stat().st_size / (1024 ** 2):.1f} MB')
display(pd.DataFrame(TIMINGS))

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except ImportError:
    print('No se detectó Colab; descargue manualmente el ZIP mostrado.')

ZIP: /content/vcmsa_paper_replica/vcmsa_replica_evidencias_20260722T143442Z.zip
Tamaño: 0.0 MB


,stage,start_utc,elapsed_seconds,peak_ram_process_tree_mb,peak_vram_process_tree_mb,peak_vram_total_device_mb,return_code,command
0,01_embeddings,2026-07-22T14:35:10.953185+00:00,47.839,8688.82,4804.0,4807.0,0,/usr/bin/python3 /content/vcmsa_paper_replica/...
1,02_vcmsa_alignment,2026-07-22T14:35:59.075852+00:00,30.847,2195.93,0.0,3.0,0,/usr/bin/python3 /content/vcmsa_paper_replica/...
2,04_codigo_propio,2026-07-22T14:36:30.955502+00:00,0.934,NaN,NaN,NaN,0,alineamiento_multiple (Python / Numba)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cómo interpretar la réplica

- Un `residues_preserved = True` demuestra que el alineador no perdió ni cambió aminoácidos.
- El `tcs_percent` permite contrastar vcMSA con el Código Propio sobre la misma familia y referencia.
- Los tiempos de Colab no deben compararse directamente con los 22.6 segundos reportados en una A100 sin indicar la GPU obtenida.
- La primera ejecución incluye descarga/carga del modelo; las repeticiones con caché pueden ser más rápidas.
- Una réplica ampliada debe repetir las fases 5–11 para varias familias y, finalmente, para las 147 familias, conservando un registro independiente por ejecución.